In [22]:
import scipy.stats as stats
import xarray
import numpy as np
import pandas as pd
import xarray
import geopandas as gpd
import pyogrio
import json
import os

In [2]:
app_workspace_dir = "../../workspaces/app_workspace"
# time range: 1940-01 ~ 2022-12
all_data = xarray.open_dataset(f"{app_workspace_dir}/combined_all_data_101.nc")
monthly_data = xarray.open_dataset(f"{app_workspace_dir}/combined_monthly_data.nc")

In [20]:
def compute_hydrosos_streamflow_layer(year, month):
    filtered_data = all_data["ds_grouped_avg"].sel(
        variable="Qout",
        time=(all_data["ds_grouped_avg"]["time"].dt.month == month) &
            (all_data["ds_grouped_avg"]["time"].dt.year == year)
    )
    month_df = filtered_data.to_dataframe().reset_index()
    average_df = monthly_data["monthly_average"].to_dataframe().reset_index()
    average_df = average_df[(average_df["variable"] == "Qout") & (average_df["month"] == month)]
    std_df = monthly_data["monthly_std_dev"].to_dataframe().reset_index()
    std_df = std_df[(std_df["variable"] == "Qout") & (std_df["month"] == month)]
    merged_df = month_df.merge(average_df[['rivid', 'monthly_average']], on='rivid', how='left').drop_duplicates(["rivid"]).reset_index()
    merged_df = merged_df.merge(std_df[['rivid', 'monthly_std_dev']], on='rivid', how='left')
    # Calculate Z-score for ds_grouped_avg using mean and standard deviation
    merged_df['z_score'] = (merged_df['ds_grouped_avg'] - merged_df['monthly_average']) / merged_df['monthly_std_dev']

    # Calculate exceedance probability using the cumulative distribution function (CDF)
    merged_df['probability'] = stats.norm.cdf(merged_df['z_score'])
    # Define the categories and corresponding colors
    categories = ["extremely dry", "dry", "normal range", "wet", "extremely wet"]

    # Map the exceedance_probability values to categories
    merged_df['classification'] = np.select(
        [merged_df['probability'] >= 0.87,
        (merged_df['probability'] >= 0.72) & (merged_df['probability'] < 0.87),
        (merged_df['probability'] >= 0.28) & (merged_df['probability'] < 0.72),
        (merged_df['probability'] >= 0.13) & (merged_df['probability'] < 0.28),
        merged_df['probability'] < 0.13],
        categories, default="unknown"
    )
    
    df_geometry = pyogrio.read_dataframe(f"{app_workspace_dir}/hydrosos_streamflow_geometry.geojson")
    merged_df = merged_df[["rivid", "classification"]].merge(df_geometry[["rivid", "strmOrder"]], how="inner")
    output_dir = f"{app_workspace_dir}/hydrosos_streamflow_by_month_no_geo"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)
    output_path = f"{app_workspace_dir}/hydrosos_streamflow_by_month_no_geo/{year}-{0 if month < 10 else ''}{month}-01.csv"
    merged_df.to_csv(output_path, index=False)
    print(f"{year}-{0 if month < 10 else ''}{month}-01 is done!")


In [21]:
compute_hydrosos_streamflow_layer(2010, 10)

2010-10-01 is done!


In [23]:
year = 2010
month = 10
filepath = f"{app_workspace_dir}/hydrosos_streamflow_by_month_no_geo/{year}-{0 if month < 10 else ''}{month}-01.csv"
df_river = pd.read_csv(filepath)
df_river

,rivid,classification,strmOrder
0,110229254,normal range,2
1,110230566,normal range,2
2,110067878,normal range,2
3,110050823,normal range,2
4,110054759,normal range,2
...,...,...,...
38164,110308113,wet,7
38165,110292368,wet,7
38166,110293680,wet,7
38167,110294992,wet,7


In [24]:
df_geometry = pyogrio.read_dataframe(f"{app_workspace_dir}/hydrosos_streamflow_geometry.geojson")
df_geometry

,rivid,strmOrder,geometry
0,110229254,2,"MULTILINESTRING ((34.69456 -5.50422, 34.70489 ..."
1,110230566,2,"MULTILINESTRING ((34.74178 -5.49356, 34.74333 ..."
2,110067878,2,"MULTILINESTRING ((34.76122 -5.49078, 34.76656 ..."
3,110050823,2,"MULTILINESTRING ((34.55044 -5.53056, 34.53778 ..."
4,110054759,2,"MULTILINESTRING ((34.56978 -5.52622, 34.56978 ..."
...,...,...,...
38164,110308113,7,"LINESTRING (40.27756 -10.58500, 40.27167 -10.5..."
38165,110292368,7,"LINESTRING (40.28889 -10.58411, 40.27756 -10.5..."
38166,110293680,7,"LINESTRING (40.35178 -10.57722, 40.34944 -10.5..."
38167,110294992,7,"LINESTRING (40.36056 -10.57000, 40.35178 -10.5..."


In [27]:
merged_df = df_river[["rivid", "strmOrder", "classification"]].merge(df_geometry[["rivid", "geometry"]], how="inner")
merged_df

,rivid,strmOrder,classification,geometry
0,110229254,2,normal range,"MULTILINESTRING ((34.69456 -5.50422, 34.70489 ..."
1,110230566,2,normal range,"MULTILINESTRING ((34.74178 -5.49356, 34.74333 ..."
2,110067878,2,normal range,"MULTILINESTRING ((34.76122 -5.49078, 34.76656 ..."
3,110050823,2,normal range,"MULTILINESTRING ((34.55044 -5.53056, 34.53778 ..."
4,110054759,2,normal range,"MULTILINESTRING ((34.56978 -5.52622, 34.56978 ..."
...,...,...,...,...
38164,110308113,7,wet,"LINESTRING (40.27756 -10.58500, 40.27167 -10.5..."
38165,110292368,7,wet,"LINESTRING (40.28889 -10.58411, 40.27756 -10.5..."
38166,110293680,7,wet,"LINESTRING (40.35178 -10.57722, 40.34944 -10.5..."
38167,110294992,7,wet,"LINESTRING (40.36056 -10.57000, 40.35178 -10.5..."


In [28]:
gdf = gpd.GeoDataFrame(merged_df)
grouped = gdf.groupby("strmOrder")
geojson_lst = []
for strmOrder, group_df in grouped:
    geojson_lst.append(group_df.to_json())
json.dumps(geojson_lst)